In [ ]:
import os
import json
import shutil
from tqdm import tqdm
from wildlife_datasets.datasets import SeaTurtleID2022
from wildlife_datasets.splits import ClosedSetSplit
from masks import rle_to_yolo

In [ ]:
root = '/data/wildlife_datasets/_smazat/SeaTurtleID2022'
SeaTurtleID2022.get_data(root)

In [ ]:
dataset = SeaTurtleID2022(root)
if dataset.df['path'].nunique() != len(dataset):
    raise ValueError('path is not unique')

In [ ]:
splitter = ClosedSetSplit(0.8)
idx_train, idx_test = splitter.split(dataset.df)[0]
idx_train += 1
idx_test += 1

In [ ]:
flipper_categories = {
    'front_left': 2,
    'front_right': 3,
    'rear_left': 4,
    'rear_right': 5,
}

root_ann = '/data/wildlife_datasets/_smazat/SeaTurtleID2022/turtles-data/data'
root_out = f'/data/wildlife_datasets/_smazat/SeaTurtleID2022_YOLO'

for addition in ['images/train', 'images/val', 'labels/train', 'labels/val']:
    os.makedirs(os.path.join(root_out, addition), exist_ok=True)

with open(os.path.join(root_ann, 'annotations.json')) as file:
    annotations = json.load(file)

In [ ]:
for i, ann_img in enumerate(annotations['images']):
    file_name = os.path.join(root_ann, ann_img['file_name'])
    if ann_img['id'] in idx_train:
        shutil.copy(file_name, f'{root_out}/images/train')
    elif ann_img['id'] in idx_test:
        shutil.copy(file_name, f'{root_out}/images/val')
    else:
        raise ValueError('Split wrong')

In [ ]:
flipper_categories[''] = 0
for ann_ann in tqdm(annotations['annotations']):
    # category_id = ann_ann['category_id'] - 1
    if ann_ann['category_id'] == 1:
        category_id = 0
    elif ann_ann['category_id'] == 3:
        category_id = 1
    else:
        location = ann_ann['attributes'].get('location', '')
        category_id = flipper_categories[location]

    image_id = ann_ann['image_id']
    rle = ann_ann['segmentation']    
    yolo_segments = rle_to_yolo(rle, class_id=category_id)
    ann_img = annotations['images'][image_id - 1]
    base_name = os.path.basename(ann_img['file_name'])
    base_name = os.path.splitext(base_name)[0] + '.txt'

    if image_id != ann_img['id']:
        raise ValueError('Image ids are not ordered')
    if ann_img['id'] in idx_train:
        file_name = f'{root_out}/labels/train/{base_name}'
    elif ann_img['id'] in idx_test:
        file_name = f'{root_out}/labels/val/{base_name}'
    else:
        raise ValueError('Split wrong')

    with open(file_name, 'a') as myfile:
        for yolo_segment in yolo_segments:
            myfile.write(yolo_segment + '\n')